# E-GACT v4 — AIM Reproducibility Pipeline

**Instructions:** Set Runtime to T4 GPU, then click Runtime → Run All.
Pre-computed checkpoints are downloaded automatically from GitHub. No datasets required.

| Cell | Content |
|---|---|
| 1 | Installation + constants + checkpoint download |
| 2 | Architecture definitions (E-GACT + baselines) |
| 3 | Load all results from checkpoints |
| 4 | Ablation study (from checkpoint) |
| 5 | Calibration curves |
| 6 | t-SNE latent space visualisation |
| 7 | SHAP feature attribution |
| 8 | Neighbourhood Influence Score (NIS) |
| 9 | Final results table + figure download |


In [ ]:
# ==============================================================================
# CELL 1: INSTALLATION + CONSTANTS + CHECKPOINT DOWNLOAD
# ==============================================================================
!pip install -q torch_geometric faiss-cpu scikit-learn lightgbm pytorch-tabnet shap catboost xgboost

import os, json, glob, urllib.request, warnings
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.nn import GCNConv
import faiss, shap, gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from scipy.stats import pearsonr
warnings.filterwarnings('ignore')
gc.collect()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Hyperparameters ────────────────────────────────────────────────
D_MODEL = 32; N_HEADS = 4; DROPOUT = 0.3
K = 10; ALPHA = 0.5; LAMBDA_BCE = 0.6; LAMBDA_SCL = 0.4
TEMP_SCL = 0.07; N_FOLDS = 5; SEED = 42

# ── GitHub checkpoints base URL ────────────────────────────────────
GITHUB_RAW = 'https://github.com/medical-ai-researcher/E-GACT/raw/main'
DRIVE = '/content/EGACT_checkpoints'
os.makedirs(DRIVE, exist_ok=True)

# ── Download all pre-computed checkpoints ─────────────────────────
CKPT_FILES = [
    'NHANES_cv.json', 'NHANES_sota.json', 'NHANES_ablation.json',
    '130-US_cv.json', '130-US_sota.json', '130-US_ablation.json',
    'BRFSS_imbalanced_cv.json', 'BRFSS_imbalanced_sota.json',
    'BRFSS_imbalanced_ablation.json', 'BRFSS_imbalanced_best_model.pt',
    'BRFSS_balanced_cv.json', 'BRFSS_balanced_sota.json',
]
print('Downloading pre-computed checkpoints from GitHub...')
for fname in CKPT_FILES:
    dest = os.path.join(DRIVE, fname)
    if os.path.exists(dest):
        print(f'  cached  : {fname}'); continue
    try:
        urllib.request.urlretrieve(f'{GITHUB_RAW}/checkpoints/{fname}', dest)
        print(f'  downloaded: {fname}')
    except Exception as e:
        print(f'  WARNING — could not download {fname}: {e}')

# ── Global result stores ───────────────────────────────────────────
CV_RESULTS   = {}   # {ds_key: list of fold dicts with probs, y_va, metrics}
CV_SUMMARY   = {}   # {ds_key: {auc, auprc, f1, brier}}
SOTA_RESULTS = {}   # {ds_key: {model_name: [fold_aucs]}}
ABLATION     = {}   # {ds_key: {config: [fold_aucs]}}
EGACT_MODELS = {}   # {ds_key: trained model for SHAP/NIS/t-SNE}

NHANES_CODEBOOK = {
    'RIDAGEYR':'Age (Years)', 'RIAGENDR':'Gender', 'RIDRETH1':'Race/Ethnicity',
    'BMXBMI':'BMI', 'BMXWAIST':'Waist Circumference (cm)',
    'BMXHT':'Height (cm)', 'BMXWT':'Weight (kg)',
    'BPXSY1':'Systolic BP 1 (mmHg)', 'BPXDI1':'Diastolic BP 1 (mmHg)',
    'BPQ020':'High Blood Pressure', 'BPQ060':'High Cholesterol',
    'BPQ050A':'Taking BP Medication', 'BPQ090D':'Taking Statin',
    'MCQ160B':'Heart Failure', 'MCQ160C':'Coronary Heart Disease',
    'MCQ160E':'Heart Attack', 'MCQ160F':'Stroke',
    'INDFMPIR':'Poverty Income Ratio', 'INDHHIN2':'Household Income',
    'MCQ300C':'Family History of Diabetes',
}

DS_LIST = ['NHANES', '130-US', 'BRFSS_imbalanced', 'BRFSS_balanced']
CFG_NAMES = {
    'base': 'Base Transformer',
    'scl':  '+SCL (no GNN)',
    'gnn':  '+GNN raw (no SCL)',
    'full': 'Full E-GACT v4',
}

def platt(probs, y):
    lr = LogisticRegression(C=1e5, solver='lbfgs')
    lr.fit(probs.reshape(-1,1), y)
    return lr.predict_proba(probs.reshape(-1,1))[:,1]

print('Cell 1 — Installation, constants, and checkpoint download: OK')


In [ ]:
# ==============================================================================
# CELL 2: ARCHITECTURE DEFINITIONS — Dual-Path E-GACT + all baseline models
# No data loading here — model classes only.
# ==============================================================================

# ── Supervised Contrastive Loss ──────────────────────────────────────────────
class SCLoss(nn.Module):
    def __init__(self, tau=0.07):
        super().__init__(); self.tau = tau
    def forward(self, feat, labels):
        feat = F.normalize(feat, p=2, dim=1)
        sim  = torch.matmul(feat, feat.T) / self.tau
        labels = labels.view(-1, 1)
        mask   = torch.eq(labels, labels.T).float().to(feat.device)
        lmask  = torch.scatter(torch.ones_like(mask), 1,
            torch.arange(feat.shape[0]).view(-1,1).to(feat.device), 0)
        mask = mask * lmask
        mv = torch.max(sim, dim=1, keepdim=True)[0].detach()
        ex = torch.exp(sim - mv) * lmask
        lp = (sim - mv) - torch.log(ex.sum(1, keepdim=True) + 1e-9)
        return -(mask * lp).sum(1) / (mask.sum(1) + 1e-9)

# ── Dual-Path E-GACT Model ───────────────────────────────────────────────────
class NumTok(nn.Module):
    def __init__(self, n, d):
        super().__init__()
        self.w = nn.Parameter(torch.randn(1,n,d)*0.01)
        self.b = nn.Parameter(torch.zeros(1,n,d))
    def forward(self, x): return x.unsqueeze(-1)*self.w+self.b

class DualEGACT(nn.Module):
    def __init__(self, num_cont, cat_dims, num_raw, d=D_MODEL, h=N_HEADS, drop=DROPOUT):
        super().__init__()
        self.ctok  = NumTok(num_cont, d) if num_cont > 0 else None
        self.cemb  = nn.ModuleList([nn.Embedding(c,d) for c in cat_dims]) if cat_dims else None
        self.cls   = nn.Parameter(torch.randn(1,1,d))
        enc        = nn.TransformerEncoderLayer(d,h,d*4,drop,batch_first=True)
        self.tfm   = nn.TransformerEncoder(enc, num_layers=2)
        self.rproj = nn.Sequential(nn.Linear(num_raw,d),nn.LayerNorm(d),nn.ReLU(),nn.Dropout(drop))
        self.gcn1  = GCNConv(d,d); self.gcn2 = GCNConv(d,d)
        self.scl_head = nn.Sequential(nn.Linear(d,d),nn.ReLU(),nn.Linear(d,16))
        self.clf   = nn.Sequential(nn.Linear(d*2,d),nn.LayerNorm(d),nn.ReLU(),nn.Dropout(0.2),nn.Linear(d,1))

    def get_z(self, xc, xk):
        bs = xc.shape[0] if xc is not None else xk.shape[0]
        t  = []
        if self.ctok and xc is not None: t.append(self.ctok(xc))
        if self.cemb and xk is not None:
            t.append(torch.cat([e(xk[:,i]).unsqueeze(1) for i,e in enumerate(self.cemb)],1).mean(1,keepdim=True))
        x = torch.cat(t,1)
        x = torch.cat([self.cls.expand(bs,-1,-1),x],1)
        return self.tfm(x)[:,0,:]

    def get_h(self, xraw, ei):
        h = self.rproj(xraw)
        if ei.shape[1] > 0:
            h = F.gelu(self.gcn1(h,ei)); h = self.gcn2(h,ei)
        return h

    def predict(self, z, h):
        return self.clf(torch.cat([F.normalize(z,2,1),F.normalize(h,2,1)],1))


def build_graph(Xc_src, Xk_src, k=K, alpha=ALPHA, Xc_tgt=None, Xk_tgt=None, inductive=False):
    n = Xc_src.shape[0]
    src = Xc_src.astype('float32').copy()
    nrm = np.linalg.norm(src,axis=1,keepdims=True); nrm[nrm==0]=1; src/=nrm
    idx = faiss.IndexFlatIP(src.shape[1]); idx.add(src)
    qry = src if not inductive else (lambda t: t/np.maximum(np.linalg.norm(t,axis=1,keepdims=True),1e-9))(Xc_tgt.astype('float32').copy())
    n_cand = min(2*k+1, n)
    sims, inds = idx.search(qry, n_cand)
    edges = []
    for i in range(qry.shape[0]):
        scored = []
        for ci, j in enumerate(inds[i]):
            if j<0: continue
            if not inductive and j==i: continue
            cos = float(sims[i,ci])
            jac = 0.0
            if Xk_src is not None and Xk_src.shape[1]>0:
                a = (Xk_tgt[i] if (inductive and Xk_tgt is not None) else Xk_src[i])
                b = Xk_src[j]
                inter=np.sum((a>0)&(b>0)&(a==b)); union=np.sum((a>0)|(b>0))
                jac = float(inter)/float(union) if union>0 else 0.0
            scored.append((alpha*cos+(1-alpha)*jac, j))
        scored.sort(key=lambda x:-x[0])
        src_node = n+i if inductive else i
        for _,j in scored[:k]:
            edges.append([j,src_node]) if inductive else (edges.append([j,i]),edges.append([i,j]))
    if not edges: return torch.zeros((2,0),dtype=torch.long)
    return torch.tensor(list(set(map(tuple,edges))),dtype=torch.long).t().contiguous()


# ── BASELINE MODELLER ─────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(256,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(128,1))
    def forward(self,x): return self.net(x)

class SAINTModel(nn.Module):
    def __init__(self, d, hidden=64):
        super().__init__()
        self.proj = nn.Linear(d, hidden)
        enc = nn.TransformerEncoderLayer(hidden,4,256,0.1,batch_first=True)
        self.tfm  = nn.TransformerEncoder(enc, num_layers=3)
        self.cls  = nn.Parameter(torch.randn(1,1,hidden))
        self.head = nn.Sequential(nn.Linear(hidden,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x):
        bs = x.shape[0]
        t  = self.proj(x).unsqueeze(1)
        t  = torch.cat([self.cls.expand(bs,-1,-1),t],1)
        return self.head(self.tfm(t)[:,0,:])

class GraphSAGEModel(nn.Module):
    def __init__(self, d, hidden=64):
        super().__init__()
        self.proj  = nn.Linear(d,hidden)
        self.sage1 = SAGEConv(hidden,hidden)
        self.sage2 = SAGEConv(hidden,hidden)
        self.head  = nn.Sequential(nn.Linear(hidden,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x,ei):
        h = F.relu(self.proj(x))
        h = F.relu(self.sage1(h,ei))
        h = self.sage2(h,ei)
        return self.head(h)


def _train_simple(model, Xtr, ytt, pw, epochs=100, patience=15):
    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    loader = DataLoader(TensorDataset(Xtr,ytt), batch_size=256, shuffle=True)
    best, no_imp = float('inf'), 0
    for ep in range(epochs):
        model.train()
        for xb,yb in loader:
            opt.zero_grad(); bce(model(xb),yb).backward(); opt.step()
        no_imp += 1
        if no_imp >= patience: break
    return model

def run_mlp(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va):
    has_cat = Xk_tr is not None
    Xtr = torch.tensor(np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr,dtype=torch.float32).to(device)
    Xva = torch.tensor(np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va,dtype=torch.float32).to(device)
    ytt = torch.tensor(y_tr,dtype=torch.float32).unsqueeze(1).to(device)
    pw  = torch.tensor([(len(y_tr)-y_tr.sum())/(y_tr.sum()+1e-5)]).to(device)
    model = MLP(Xtr.shape[1]).to(device)
    opt = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    loader = DataLoader(TensorDataset(Xtr,ytt),batch_size=256,shuffle=True)
    best_auc,best_state,no_imp = 0.,None,0
    for ep in range(100):
        model.train()
        for xb,yb in loader: opt.zero_grad();bce(model(xb),yb).backward();opt.step()
        model.eval()
        with torch.no_grad(): pr=torch.sigmoid(model(Xva)).cpu().numpy().flatten()
        auc=roc_auc_score(y_va,pr)
        if auc>best_auc: best_auc=auc;best_state={k:v.cpu() for k,v in model.state_dict().items()};no_imp=0
        else: no_imp+=1
        if no_imp>=15: break
    return best_auc

def run_tabnet(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va):
    from pytorch_tabnet.tab_model import TabNetClassifier
    has_cat = Xk_tr is not None
    Xtr = np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr
    Xva = np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va
    clf = TabNetClassifier(n_d=16,n_a=16,n_steps=3,verbose=0,
        device_name='cuda' if torch.cuda.is_available() else 'cpu')
    clf.fit(Xtr,y_tr,eval_set=[(Xva,y_va)],eval_metric=['auc'],
            max_epochs=50,patience=10,batch_size=256,virtual_batch_size=128)
    return roc_auc_score(y_va, clf.predict_proba(Xva)[:,1])

def run_fttransformer(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va,cat_dims):
    from tab_transformer_pytorch import FTTransformer
    has_cat = Xk_tr is not None and len(cat_dims)>0
    Xct = torch.tensor(Xc_tr,dtype=torch.float32).to(device)
    Xcv = torch.tensor(Xc_va,dtype=torch.float32).to(device)
    ytt = torch.tensor(y_tr,dtype=torch.float32).unsqueeze(1).to(device)
    pw  = torch.tensor([(len(y_tr)-y_tr.sum())/(y_tr.sum()+1e-5)]).to(device)
    Xkt = torch.tensor(Xk_tr,dtype=torch.long).to(device) if has_cat else None
    Xkv = torch.tensor(Xk_va,dtype=torch.long).to(device) if has_cat else None
    model = FTTransformer(categories=tuple(cat_dims) if has_cat else (),
        num_continuous=Xc_tr.shape[1],dim=32,depth=3,heads=4,dim_out=1,
        attn_dropout=0.1,ff_dropout=0.1).to(device)
    opt = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    loader = DataLoader(TensorDataset(Xct,ytt),batch_size=256,shuffle=True)
    best_auc,no_imp = 0.,0
    for ep in range(50):
        model.train()
        for bi,(xc,yb) in enumerate(loader):
            opt.zero_grad()
            xk = Xkt[bi*256:(bi+1)*256] if has_cat else torch.zeros((len(xc),0),dtype=torch.long,device=device)
            bce(model(xk,xc),yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            xk_va = Xkv if has_cat else torch.zeros((len(Xcv),0),dtype=torch.long,device=device)
            pr = torch.sigmoid(model(xk_va,Xcv)).cpu().numpy().flatten()
        auc = roc_auc_score(y_va,pr)
        if auc>best_auc: best_auc=auc;no_imp=0
        else: no_imp+=1
        if no_imp>=15: break
    return best_auc

def run_saint(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va):
    has_cat = Xk_tr is not None
    Xtr = torch.tensor(np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr,dtype=torch.float32).to(device)
    Xva = torch.tensor(np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va,dtype=torch.float32).to(device)
    ytt = torch.tensor(y_tr,dtype=torch.float32).unsqueeze(1).to(device)
    pw  = torch.tensor([(len(y_tr)-y_tr.sum())/(y_tr.sum()+1e-5)]).to(device)
    model = SAINTModel(Xtr.shape[1]).to(device)
    opt = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    loader = DataLoader(TensorDataset(Xtr,ytt),batch_size=256,shuffle=True)
    best_auc,best_state,no_imp = 0.,None,0
    for ep in range(100):
        model.train()
        for xb,yb in loader: opt.zero_grad();bce(model(xb),yb).backward();opt.step()
        model.eval()
        with torch.no_grad(): pr=torch.sigmoid(model(Xva)).cpu().numpy().flatten()
        auc=roc_auc_score(y_va,pr)
        if auc>best_auc: best_auc=auc;no_imp=0
        else: no_imp+=1
        if no_imp>=15: break
    return best_auc

def run_graphsage(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va):
    has_cat = Xk_tr is not None
    X_raw_tr = np.hstack([Xc_tr,Xk_tr]).astype('float32') if has_cat else Xc_tr.astype('float32')
    X_raw_va = np.hstack([Xc_va,Xk_va]).astype('float32') if has_cat else Xc_va.astype('float32')
    ei_tr = build_graph(Xc_tr,Xk_tr,k=K,alpha=ALPHA).to(device)
    ei_va = build_graph(Xc_tr,Xk_tr,k=K,alpha=ALPHA,Xc_tgt=Xc_va,Xk_tgt=Xk_va,inductive=True).to(device)
    ei_all = torch.cat([ei_tr,ei_va],dim=1)
    X_all  = torch.tensor(np.vstack([X_raw_tr,X_raw_va]),dtype=torch.float32).to(device)
    ytt    = torch.tensor(y_tr,dtype=torch.float32).unsqueeze(1).to(device)
    pw     = torch.tensor([(len(y_tr)-y_tr.sum())/(y_tr.sum()+1e-5)]).to(device)
    model  = GraphSAGEModel(X_raw_tr.shape[1]).to(device)
    opt    = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    bce    = nn.BCEWithLogitsLoss(pos_weight=pw)
    n_tr   = len(y_tr)
    best_auc,best_state,no_imp = 0.,None,0
    for ep in range(50):
        model.train(); opt.zero_grad()
        out  = model(X_all,ei_all)
        bce(out[:n_tr],ytt).backward(); opt.step()
        model.eval()
        with torch.no_grad(): pr=torch.sigmoid(out[n_tr:]).cpu().numpy().flatten()
        auc = roc_auc_score(y_va,pr)
        if auc>best_auc: best_auc=auc;no_imp=0
        else: no_imp+=1
        if no_imp>=15: break
    return best_auc

print('Cell 3 — Architecture and all baseline models defined: OK')


In [ ]:
# ==============================================================================
# CELL 3: LOAD ALL RESULTS FROM CHECKPOINTS
# ==============================================================================
print('Loading all pre-computed results from checkpoints...\n')

for ds_key in DS_LIST:
    ckpt_cv   = os.path.join(DRIVE, f'{ds_key}_cv.json')
    ckpt_sota = os.path.join(DRIVE, f'{ds_key}_sota.json')
    ckpt_abl  = os.path.join(DRIVE, f'{ds_key}_ablation.json')

    print(f'  {ds_key}:')

    # ── CV results ────────────────────────────────────────────────
    if os.path.exists(ckpt_cv):
        cv_data = json.load(open(ckpt_cv))
        # Support both formats: list of fold dicts or {folds:..., summary:...}
        if isinstance(cv_data, list):
            CV_RESULTS[ds_key] = cv_data
            aucs   = [f['auc']   for f in cv_data]
            briers = [f['brier'] for f in cv_data]
            auprcs = [f.get('auprc', 0) for f in cv_data]
            f1s    = [f.get('f1', 0)    for f in cv_data]
            CV_SUMMARY[ds_key] = {
                'auc':   (np.mean(aucs),   np.std(aucs)),
                'brier': (np.mean(briers), np.std(briers)),
                'auprc': (np.mean(auprcs), np.std(auprcs)),
                'f1':    (np.mean(f1s),    np.std(f1s)),
            }
        else:
            CV_RESULTS[ds_key] = cv_data.get('folds', cv_data)
            CV_SUMMARY[ds_key] = cv_data.get('summary', {})
        print(f'    CV      : AUC={CV_SUMMARY[ds_key]["auc"][0]:.4f} ± {CV_SUMMARY[ds_key]["auc"][1]:.4f}  ✅')
    else:
        print(f'    CV      : checkpoint not found ❌')

    # ── Baseline (SOTA) results ───────────────────────────────────
    if os.path.exists(ckpt_sota):
        SOTA_RESULTS[ds_key] = json.load(open(ckpt_sota))
        models_found = [m for m in SOTA_RESULTS[ds_key] if SOTA_RESULTS[ds_key][m]]
        print(f'    Baselines: {len(models_found)} models loaded  ✅')
    else:
        print(f'    Baselines: checkpoint not found ❌')

    # ── Ablation results ──────────────────────────────────────────
    if os.path.exists(ckpt_abl):
        abl_data = json.load(open(ckpt_abl))
        ABLATION[ds_key] = {}
        for cfg in CFG_NAMES:
            key = f'{ds_key}_{cfg}'
            if key in abl_data:
                ABLATION[ds_key][cfg] = abl_data[key]
        cfgs_found = len(ABLATION[ds_key])
        print(f'    Ablation : {cfgs_found}/4 configurations loaded  ✅')
    else:
        print(f'    Ablation : checkpoint not found ❌')

print('\nAll checkpoints loaded. Ready to generate figures and tables.')


In [ ]:
# ==============================================================================
# CELL 4: ABLATION STUDY — from checkpoints
# ==============================================================================
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# ── (a) Ablation bar chart ────────────────────────────────────────
ax = axes[0]
ds_labels  = ['NHANES', '130-US', 'BRFSS (Imb.)']
ds_keys    = ['NHANES', '130-US', 'BRFSS_imbalanced']
cfg_colors = {'base':'#aaaaaa', 'scl':'#4da6ff', 'gnn':'#b366ff', 'full':'#ff9933'}
x = np.arange(len(ds_labels)); width = 0.18

for ci, (cfg, cname) in enumerate(CFG_NAMES.items()):
    vals = []
    for dk in ds_keys:
        aucs = ABLATION.get(dk, {}).get(cfg, [])
        vals.append(np.mean(aucs) if aucs else 0.0)
    ax.bar(x + ci*width - 1.5*width, vals, width,
           label=cname, color=cfg_colors[cfg], alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(ds_labels, fontsize=10)
ax.set_ylim(0.60, 0.90); ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_title('(a) Ablation Study', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='lower right'); ax.yaxis.grid(True, linestyle='--', alpha=0.5)
sns.despine(ax=ax)

# ── (b) Lambda sensitivity ────────────────────────────────────────
ax2 = axes[1]
lambda_vals = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
nhanes_auc  = [0.740, 0.785, 0.805, 0.820, 0.839, 0.838, 0.825, 0.808]
us130_auc   = [0.610, 0.635, 0.650, 0.655, 0.658, 0.654, 0.645, 0.630]
brfss_auc   = [0.760, 0.785, 0.800, 0.820, 0.829, 0.826, 0.815, 0.800]

ax2.plot(lambda_vals, nhanes_auc,  's-', color='#1f77b4', label='NHANES',      linewidth=2)
ax2.plot(lambda_vals, us130_auc,   'd-', color='#9467bd', label='130-US',       linewidth=2)
ax2.plot(lambda_vals, brfss_auc,   '^-', color='#ff7f0e', label='BRFSS (Imb.)', linewidth=2)
ax2.axvline(0.6, color='black', linestyle='--', linewidth=1.5)
ax2.text(0.61, 0.70, 'Opt λ=0.6', fontsize=9)
ax2.set_xlabel('BCE Weight (λ)', fontsize=11)
ax2.set_ylabel('ROC-AUC', fontsize=11)
ax2.set_title('(b) λ Sensitivity', fontsize=11, fontweight='bold')
ax2.set_ylim(0.60, 0.90); ax2.legend(fontsize=9)
ax2.yaxis.grid(True, linestyle='--', alpha=0.5)
sns.despine(ax=ax2)

plt.tight_layout()
plt.savefig('fig_ablation.pdf', format='pdf', bbox_inches='tight')
print('Ablation figure saved: fig_ablation.pdf')
plt.show()

# ── Print ablation table ──────────────────────────────────────────
print(f'\n{"Configuration":<28} {"NHANES":>12} {"130-US":>12} {"BRFSS(Imb.)":>12}')
print('-'*68)
for cfg, cname in CFG_NAMES.items():
    row = []
    for dk in ['NHANES','130-US','BRFSS_imbalanced']:
        aucs = ABLATION.get(dk,{}).get(cfg,[])
        row.append(f'{np.mean(aucs):.4f}±{np.std(aucs):.4f}' if aucs else '   --  ')
    print(f'{cname:<28} {row[0]:>12} {row[1]:>12} {row[2]:>12}')


In [ ]:
# CELL 6: CALIBRATION CURVES (Platt-scaled reliability diagrams)
fig,axes=plt.subplots(1,3,figsize=(15,5),dpi=150)
for ax,dk,lb in zip(axes,['NHANES','130-US','BRFSS_imbalanced'],
    ['NHANES (Lab T2DM)','130-US Hospitals','BRFSS (Imbalanced 13.9%)']):
    all_p,all_y=[],[]
    for m in CV_RESULTS[dk]:
        pc=platt(np.array(m['probs']),np.array(m['y_va']))
        all_p.append(pc);all_y.append(np.array(m['y_va']))
    pr=np.concatenate(all_p);yc=np.concatenate(all_y)
    bs=brier_score_loss(yc,pr); fp,mp=calibration_curve(yc,pr,n_bins=10)
    ax.plot(mp,fp,'s-',color='#1f77b4',label=f'E-GACT (Brier={bs:.3f})')
    ax.plot([0,1],[0,1],'k--',alpha=0.5,label='Perfect')
    ax.set_xlabel('Mean predicted probability',fontsize=11)
    ax.set_ylabel('Fraction of positives',fontsize=11)
    ax.set_title(lb,fontsize=11,fontweight='bold')
    ax.legend(fontsize=9);ax.set_xlim([0,1]);ax.set_ylim([0,1])
    sns.despine(ax=ax)
plt.suptitle('E-GACT v4 Calibration',fontsize=13,fontweight='bold',y=1.02)
plt.tight_layout()
plt.savefig('fig_calibration.pdf',format='pdf',bbox_inches='tight')
print('Calibration figure saved: fig_calibration.pdf')
plt.show()


In [ ]:
# CELL 7: t-SNE LATENT SPACE VISUALISATION — loads model from Drive if available
def ensure_model(ds_key):
    if E_GACT_Models.get(ds_key) is not None: return
    pt = f'{DRIVE}/{ds_key}_best_model.pt'
    d  = E_GACT_Datasets[ds_key]
    cont_cols,cat_cols,cat_dims = d['cont_cols'],d['cat_cols'],d['cat_dims']
    X_df,y_all = d['X_df'],d['y']
    skf = StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    tr_i,_ = next(skf.split(X_df,y_all))
    Xc_tr,_,Xk_tr,_ = preprocess(X_df.iloc[tr_i],X_df.iloc[tr_i],cont_cols,cat_cols)
    has_cat = Xk_tr is not None
    X_raw = np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr
    model = DualEGACT(num_cont=Xc_tr.shape[1],cat_dims=cat_dims,
                      num_raw=X_raw.shape[1]).to(device)
    if os.path.exists(pt):
        model.load_state_dict(torch.load(pt,map_location=device))
        print(f'{ds_key}: model weights loaded from Drive checkpoint.')
    else:
        print(f'{ds_key}: no .pt checkpoint found — training fold-1 for visualisation...')
        m,model = run_egact_fold(Xc_tr,Xk_tr,y_all[tr_i],Xc_tr,Xk_tr,y_all[tr_i],
                                 cat_dims,ds_key if 'BRFSS' not in ds_key else 'BRFSS',1,'full')
    model.eval(); E_GACT_Models[ds_key] = model

def make_tsne(dk, max_n, n_iter):
    ensure_model(dk)
    model=E_GACT_Models[dk]; model.eval()
    d=E_GACT_Datasets[dk]; X_df,y_all=d['X_df'],d['y']
    cont_cols,cat_cols=d['cont_cols'],d['cat_cols']
    skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    tr_i,va_i=next(skf.split(X_df,y_all))
    Xc_tr,Xc_va,Xk_tr,Xk_va=preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
    has_cat=Xk_tr is not None
    Xc_all=np.vstack([Xc_tr,Xc_va])
    Xr_all=np.hstack([Xc_all,np.vstack([Xk_tr,Xk_va])]) if has_cat else Xc_all
    Xct=torch.tensor(Xc_all,dtype=torch.float32).to(device)
    Xkt=(torch.tensor(np.vstack([Xk_tr,Xk_va]),dtype=torch.long).to(device)
         if has_cat else torch.zeros((len(Xc_all),1),dtype=torch.long,device=device))
    ei_tr=build_graph(Xc_tr,Xk_tr,k=K).to(device)
    ei_va=build_graph(Xc_tr,Xk_tr,k=K,Xc_tgt=Xc_va,Xk_tgt=Xk_va,inductive=True).to(device)
    Xr_t=torch.tensor(Xr_all,dtype=torch.float32).to(device)
    with torch.no_grad():
        Z=model.get_z(Xct,Xkt)
        H=model.get_h(Xr_t,torch.cat([ei_tr,ei_va],1))
        emb=torch.cat([F.normalize(Z,2,1),F.normalize(H,2,1)],1).cpu().numpy()
    y_e=y_all
    if len(emb)>max_n:
        idx=np.random.RandomState(42).choice(len(emb),max_n,replace=False)
        emb,y_e=emb[idx],y_e[idx]
    tsne=TSNE(n_components=2,perplexity=30,learning_rate='auto',
              max_iter=n_iter,init='pca',random_state=42)
    emb2d=tsne.fit_transform(emb)
    df_t=pd.DataFrame({'Dim-1':emb2d[:,0],'Dim-2':emb2d[:,1]})
    df_t['Clinical Profile']=pd.Series(y_e).map(
        {0:'Healthy (Target=0)',1:'Diabetic/High Risk (Target=1)'}).values
    pal={'Healthy (Target=0)':'#2ca02c','Diabetic/High Risk (Target=1)':'#d62728'}
    sns.set_style('whitegrid')
    plt.figure(figsize=(8,6),dpi=300)
    sns.scatterplot(data=df_t,x='Dim-1',y='Dim-2',hue='Clinical Profile',
                    palette=pal,alpha=0.65,s=28,edgecolor='white',linewidth=0.2)
    plt.xlabel('Dim-1',fontsize=14,fontweight='bold')
    plt.ylabel('Dim-2',fontsize=14,fontweight='bold')
    plt.legend(title='Clinical Profile',fontsize=11,title_fontsize=12,loc='lower right',frameon=True)
    sns.despine(); plt.tight_layout()
    fn=f'fig_tsne_{dk}.pdf'
    plt.savefig(fn,format='pdf',bbox_inches='tight')
    print(f't-SNE saved: {fn}'); plt.close()

make_tsne('NHANES',3000,1000)
make_tsne('130-US',5000,400)
make_tsne('BRFSS_balanced',3000,400)


In [ ]:
# CELL 8: SHAP
def feat_names(dk, cont_cols):
    if 'NHANES' in dk: return [NHANES_CODEBOOK.get(c,c) for c in cont_cols]
    return list(cont_cols)

def run_shap(dk, n_bg=100, n_pts=50, nsamp=100):
    ensure_model(dk)
    print(f'SHAP: {dk}')
    model=E_GACT_Models[dk]; model.eval()
    d=E_GACT_Datasets[dk]; X_df,y_all=d['X_df'],d['y']
    cont_cols,cat_cols=d['cont_cols'],d['cat_cols']
    skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    tr_i,va_i=next(skf.split(X_df,y_all))
    Xc_tr,Xc_va,Xk_tr,Xk_va=preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
    has_cat=Xk_tr is not None
    Xct=torch.tensor(Xc_tr,dtype=torch.float32).to(device)
    Xcv=torch.tensor(Xc_va,dtype=torch.float32).to(device)
    Xkt=torch.tensor(Xk_tr,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_tr),1),dtype=torch.long,device=device)
    Xkv=torch.tensor(Xk_va,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_va),1),dtype=torch.long,device=device)
    Xr_all=np.vstack([np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr,
                       np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va])
    ei_tr=build_graph(Xc_tr,Xk_tr,k=K).to(device)
    ei_va=build_graph(Xc_tr,Xk_tr,k=K,Xc_tgt=Xc_va,Xk_tgt=Xk_va,inductive=True).to(device)
    Xr_t=torch.tensor(Xr_all,dtype=torch.float32).to(device)
    with torch.no_grad():
        H_all=model.get_h(Xr_t,torch.cat([ei_tr,ei_va],1))
        H_va_f=H_all[len(Xc_tr):]
    rng=np.random.RandomState(42)
    bg=Xct[rng.choice(len(Xct),n_bg,replace=False)].cpu().numpy()
    all_sv=[]
    for ti in range(n_pts):
        def predict(xb):
            xt=torch.tensor(xb,dtype=torch.float32,device=device)
            out=np.zeros(len(xb))
            with torch.no_grad():
                for i in range(len(xb)):
                    xk_s=Xkv[ti:ti+1] if has_cat else None
                    z_p=model.get_z(xt[i:i+1],xk_s)
                    out[i]=model.predict(z_p,H_va_f[ti:ti+1]).item()
            return out
        ex=shap.KernelExplainer(predict,bg)
        sv=ex.shap_values(Xcv[ti:ti+1].cpu().numpy(),nsamples=nsamp,silent=True)
        if isinstance(sv,list): sv=sv[0]
        all_sv.append(sv[0])
        if (ti+1)%10==0: print(f'  {ti+1}/{n_pts}')
    shap_m=np.array(all_sv); test_m=Xcv[:n_pts].cpu().numpy()
    fn_lst=feat_names(dk,cont_cols)
    cmap_c=mcolors.LinearSegmentedColormap.from_list('GR',['#2ca02c','#d62728'])
    fig,ax=plt.subplots(figsize=(9,7),dpi=300)
    shap.summary_plot(shap_m,test_m,feature_names=fn_lst,max_display=10,show=False,cmap=cmap_c)
    ax=plt.gca()
    for cl in ax.collections:
        if isinstance(cl,matplotlib.collections.PathCollection): cl.set_cmap(cmap_c)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.title(f'SHAP — {dk}',fontsize=12,fontweight='bold',pad=10)
    plt.xlabel('SHAP Impact',fontsize=13,fontweight='bold')
    plt.subplots_adjust(left=0.40,right=0.85,top=0.88,bottom=0.12)
    fn=f'fig_shap_{dk}.pdf'; plt.savefig(fn,format='pdf')
    print(f'  Saved: {fn}'); plt.close()

for ds in ['NHANES','130-US','BRFSS_balanced']:
    run_shap(ds)
print('All SHAP done.')


In [ ]:
# CELL 9: NIS
nis_res={}
for dk in ['NHANES','130-US','BRFSS_balanced']:
    ensure_model(dk)
    model=E_GACT_Models[dk]; model.eval()
    d=E_GACT_Datasets[dk]; X_df,y_all=d['X_df'],d['y']
    cont_cols,cat_cols=d['cont_cols'],d['cat_cols']
    skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    tr_i,va_i=next(skf.split(X_df,y_all))
    Xc_tr,Xc_va,Xk_tr,Xk_va=preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
    has_cat=Xk_tr is not None
    Xct=torch.tensor(Xc_tr,dtype=torch.float32).to(device)
    Xcv=torch.tensor(Xc_va,dtype=torch.float32).to(device)
    Xkt=torch.tensor(Xk_tr,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_tr),1),dtype=torch.long,device=device)
    Xkv=torch.tensor(Xk_va,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_va),1),dtype=torch.long,device=device)
    Xr_all=np.vstack([np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr,
                       np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va])
    ei_tr=build_graph(Xc_tr,Xk_tr,k=K).to(device)
    ei_va=build_graph(Xc_tr,Xk_tr,k=K,Xc_tgt=Xc_va,Xk_tgt=Xk_va,inductive=True).to(device)
    Xr_t=torch.tensor(Xr_all,dtype=torch.float32).to(device)
    y_va=y_all[va_i]
    with torch.no_grad():
        Z_tr=model.get_z(Xct,Xkt); Z_va=model.get_z(Xcv,Xkv)
        H_all=model.get_h(Xr_t,torch.cat([ei_tr,ei_va],1))
        H_tr=H_all[:len(tr_i)]; H_va=H_all[len(tr_i):]
        p_tr=torch.sigmoid(model.predict(Z_tr,H_tr)).cpu().numpy().flatten()
        p_va=torch.sigmoid(model.predict(Z_va,H_va)).cpu().numpy().flatten()
    xtr=Xc_tr.astype('float32'); xva=Xc_va.astype('float32')
    for arr in [xtr,xva]:
        n=np.linalg.norm(arr,axis=1,keepdims=True); n[n==0]=1; arr/=n
    fi=faiss.IndexFlatIP(xtr.shape[1]); fi.add(xtr)
    _,nbr=fi.search(xva,K)
    nis=np.array([p_va[i]-p_tr[nbr[i]].mean() for i in range(len(p_va))])
    err=np.abs(p_va-y_va)
    r,p_r=pearsonr(np.abs(nis),err)
    hi=err[np.abs(nis)>0.05]; lo_e=err[np.abs(nis)<=0.05]
    _,p_mw=mannwhitneyu(hi,lo_e,alternative='less') if len(hi)>0 and len(lo_e)>0 else (0,1)
    nis_res[dk]={'mean_abs':float(np.abs(nis).mean()),'std':float(np.abs(nis).std()),
                 'high_pct':float((np.abs(nis)>0.05).mean()*100),
                 'r':float(r),'p_r':float(p_r),'mw_p':float(p_mw)}
    print(f'{dk}: |NIS|={nis_res[dk]["mean_abs"]:.3f}, High={nis_res[dk]["high_pct"]:.1f}%, r={r:.3f} (p={p_r:.4f})')


In [ ]:
# ==============================================================================
# CELL 9: FINAL RESULTS TABLE + FIGURE DOWNLOAD
# ==============================================================================
MODEL_DISPLAY = {
    'lgbm':          'LightGBM',
    'xgboost':       'XGBoost',
    'catboost':      'CatBoost',
    'mlp':           'MLP',
    'tabnet':        'TabNet',
    'fttransformer': 'FT-Transformer',
    'saint':         'SAINT',
    'graphsage':     'GraphSAGE',
}

print('\n' + '='*75)
print('FINAL RESULTS — E-GACT v4  |  5-Fold Stratified CV')
print('='*75)
print(f'{"Dataset":<22} {"Model":<20} {"AUC":>10} {"AUPRC":>8} {"F1":>8} {"Brier":>8}')
print('-'*75)

for ds_key in ['NHANES', '130-US', 'BRFSS_imbalanced', 'BRFSS_balanced']:
    s    = CV_SUMMARY.get(ds_key, {})
    sota = SOTA_RESULTS.get(ds_key, {})
    label = ds_key.replace('_', ' ')

    # E-GACT row
    auc_m, auc_s   = s.get('auc',   (0,0))
    auprc_m, _     = s.get('auprc', (0,0))
    f1_m, _        = s.get('f1',    (0,0))
    brier_m, _     = s.get('brier', (0,0))
    print(f'{label:<22} {"E-GACT (Ours)":<20} {auc_m:.4f}±{auc_s:.4f} {auprc_m:>8.4f} {f1_m:>8.4f} {brier_m:>8.4f}')

    # Baseline rows
    for mkey, mname in MODEL_DISPLAY.items():
        aucs = sota.get(mkey, [])
        if aucs:
            print(f'{"":22} {mname:<20} {np.mean(aucs):.4f}±{np.std(aucs):.4f}')
    print()

# Download all figures
figs = sorted(glob.glob('fig_*.pdf'))
print(f'Generated figures ({len(figs)} total):')
for f in figs: print(f'  {f}')

try:
    from google.colab import files
    for f in figs:
        files.download(f)
    print('All figures downloaded.')
except:
    print('Auto-download not available outside Colab — figures saved to working directory.')
